# M3.2 · MLP ve matris formunda backprop

İlk gerçek sinir ağı. 2-2-1'lik bir ağda ileri ve geri geçişi elle yapıyor, saf Python'da katman katman forward/backward fonksiyonları yazıyor, NumPy'da CS231n tarzı modüler katmanlarla spiral veriyi ayıran iki katmanlı bir ağ eğitiyor, PyTorch'ta gradyanlarını autograd ile doğruluyor ve en sonda `torch.func` ile 16 ağı tek ileri geçişte (model ensembling) koşturuyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-2-mlp-backprop.html](https://m-gonul.github.io/pytorch-lab/m3-2-mlp-backprop.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_2_mlp_backprop.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def hand():\n        L, g = need(ns, \"mlp_loss\")([1, 2], 1, [[1, 2], [-1, 1]], [0, 0], [1, -2], 0)\n        close(L, 2); close(g[\"W1\"], [[2, 4], [-4, -8]]); close(g[\"w2\"], [10, 2]); close(g[\"x\"], [6, 0]); close(g[\"b1\"], [2, -4]); close(g[\"b2\"], 2)\n    R.test(\"elle örnek: tüm gradyanlar\", hand)\n\n    def num():\n        rng = random.Random(0)\n        x = [rng.gauss(0, 1) for _ in range(3)]\n        W1 = [[rng.gauss(0, 1) for _ in range(3)] for _ in range(4)]\n        b1 = [rng.gauss(0, 0.5) for _ in range(4)]\n        w2 = [rng.gauss(0, 1) for _ in range(4)]\n        f = lambda W1_, b1_, w2_, b2_, x_: need(ns, \"mlp_loss\")(x_, 0.7, W1_, b1_, w2_, b2_)[0]\n        L, g = need(ns, \"mlp_loss\")(x, 0.7, W1, b1, w2, 0.1)\n        h = 1e-6\n        for i in range(4):\n            for j in range(3):\n                Wp = [r[:] for r in W1]; Wm = [r[:] for r in W1]\n                Wp[i][j] += h; Wm[i][j] -= h\n                close(g[\"W1\"][i][j], (f(Wp, b1, w2, 0.1, x) - f(Wm, b1, w2, 0.1, x)) / (2 * h), rtol=1e-5, atol=1e-7, what=f\"dW1[{i}][{j}]\")\n        for j in range(3):\n            xp, xm = x[:], x[:]; xp[j] += h; xm[j] -= h\n            close(g[\"x\"][j], (f(W1, b1, w2, 0.1, xp) - f(W1, b1, w2, 0.1, xm)) / (2 * h), rtol=1e-5, atol=1e-7, what=f\"dx[{j}]\")\n    R.test(\"3-4-1 ağ: sayısal gradyanla aynı\", num)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef spiral(n=100, K=3, seed=0):\n    rng = np.random.default_rng(seed)\n    X, y = [], []\n    for k in range(K):\n        r = np.linspace(0.0, 1, n)\n        t = np.linspace(k * 4, (k + 1) * 4, n) + rng.normal(size=n) * 0.2\n        X.append(np.c_[r * np.sin(t), r * np.cos(t)]); y += [k] * n\n    return np.concatenate(X), np.array(y)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    for fn in (\"affine_forward\", \"affine_backward\", \"relu_forward\", \"relu_backward\", \"softmax_loss\", \"two_layer_loss\"):\n        R.test(f\"{fn}: döngü yok\", lambda fn=fn: max_loops(src, 0, func=fn))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n\n    def aff():\n        x, w, b = rng.normal(size=(5, 4)), rng.normal(size=(4, 3)), rng.normal(size=3)\n        out, cache = need(ns, \"affine_forward\")(x, w, b)\n        close(out, x @ w + b)\n        dout = rng.normal(size=(5, 3))\n        dx, dw, db = need(ns, \"affine_backward\")(dout, cache)\n        close(dx, dout @ w.T); close(dw, x.T @ dout); close(db, dout.sum(0))\n    R.test(\"affine forward/backward\", aff)\n\n    def grads():\n        X, y = rng.normal(size=(7, 3)), rng.integers(0, 4, 7)\n        p = {\"W1\": rng.normal(size=(3, 5)), \"b1\": rng.normal(size=5) * 0.1, \"W2\": rng.normal(size=(5, 4)), \"b2\": rng.normal(size=4) * 0.1}\n        L, g = need(ns, \"two_layer_loss\")(X, y, p, reg=0.05)\n        h = 1e-6\n        for k in p:\n            flat = p[k].reshape(-1)\n            for idx in rng.choice(flat.size, min(4, flat.size), replace=False):\n                old = flat[idx]\n                flat[idx] = old + h; Lp = need(ns, \"two_layer_loss\")(X, y, p, 0.05)[0]\n                flat[idx] = old - h; Lm = need(ns, \"two_layer_loss\")(X, y, p, 0.05)[0]\n                flat[idx] = old\n                close(g[k].reshape(-1)[idx], (Lp - Lm) / (2 * h), rtol=1e-4, atol=1e-7, what=f\"d{k}[{idx}]\")\n    R.test(\"two_layer_loss: sayısal gradyanla aynı (reg dahil)\", grads)\n\n    def tr():\n        X, y = spiral()\n        p = need(ns, \"train\")(X, y, 100, 3)\n        h = np.maximum(0, X @ p[\"W1\"] + p[\"b1\"])\n        acc = ((h @ p[\"W2\"] + p[\"b2\"]).argmax(1) == y).mean()\n        ensure(acc >= 0.95, f\"spiral doğruluğu {acc:.2f}\")\n        return f\"spiral (3 kol): eğitim doğruluğu {acc:.3f}\"\n    R.test(\"spiral veride eğitim ≥ 0.95\", tr)\n",
 "d": "import math\nimport torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef spiral(n=100, K=3, seed=0):\n    g = torch.Generator().manual_seed(seed)\n    X, y = [], []\n    for k in range(K):\n        r = torch.linspace(0.0, 1, n)\n        t = torch.linspace(k * 4, (k + 1) * 4, n) + torch.randn(n, generator=g) * 0.2\n        X.append(torch.stack([r * t.sin(), r * t.cos()], 1)); y += [k] * n\n    return torch.cat(X), torch.tensor(y)\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n\n    def gr():\n        X = torch.randn(9, 3, generator=g, dtype=torch.float64); y = torch.randint(0, 4, (9,), generator=g)\n        W1, b1 = torch.randn(3, 6, generator=g, dtype=torch.float64), torch.randn(6, generator=g, dtype=torch.float64)\n        W2, b2 = torch.randn(6, 4, generator=g, dtype=torch.float64), torch.randn(4, generator=g, dtype=torch.float64)\n        ps = [t.clone().requires_grad_(True) for t in (W1, b1, W2, b2)]\n        L = need(ns, \"two_layer_torch\")(X, y, *ps, 0.05)\n        L.backward()\n        Z = X @ W1 + b1; H = Z.clamp(min=0); S = H @ W2 + b2\n        dS = (torch.softmax(S, 1) - torch.nn.functional.one_hot(y, 4)) / 9\n        dW2 = H.T @ dS + 0.1 * W2; db2 = dS.sum(0)\n        dZ = (dS @ W2.T) * (Z > 0)\n        dW1 = X.T @ dZ + 0.1 * W1; db1 = dZ.sum(0)\n        for p, want, n in zip(ps, (dW1, db1, dW2, db2), (\"W1\", \"b1\", \"W2\", \"b2\")):\n            close(p.grad, want, what=f\"d{n}\")\n        return \"autograd = elle türetilen matris formülleri\"\n    R.test(\"autograd gradyanları matris formülleriyle aynı\", gr)\n\n    def tr():\n        X, y = spiral()\n        torch.manual_seed(0)\n        m = need(ns, \"train_mlp\")(X, y, 100, 3)\n        ensure(isinstance(m, nn.Module), \"nn.Module bekleniyordu\")\n        acc = (m(X).argmax(1) == y).float().mean().item()\n        ensure(acc >= 0.95, f\"spiral doğruluğu {acc:.2f}\")\n        return f\"spiral: doğruluk {acc:.3f}\"\n    R.test(\"train_mlp spiral veride ≥ 0.95\", tr)\n",
 "e": "import torch\nfrom torch import nn\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"ensemble fonksiyonlarında döngü yok\", lambda: (max_loops(src, 0, func=\"ensemble_logits\"), max_loops(src, 0, func=\"ensemble_predict\")))\n    torch.manual_seed(0)\n    models = [nn.Sequential(nn.Linear(4, 24), nn.ReLU(), nn.Linear(24, 5)) for _ in range(8)]\n    X = torch.randn(50, 4)\n    ref = torch.stack([m(X) for m in models])\n\n    def lg():\n        out = need(ns, \"ensemble_logits\")(models, X)\n        ensure(tuple(out.shape) == (8, 50, 5), f\"shape {tuple(out.shape)}\")\n        close(out, ref, rtol=1e-5, atol=1e-6)\n    R.test(\"ensemble_logits = modeller üzerinde döngü\", lg)\n    R.test(\"ensemble_predict: ortalama olasılık argmax\", lambda: ensure(torch.equal(need(ns, \"ensemble_predict\")(models, X), ref.softmax(-1).mean(0).argmax(-1)), \"tahminler farklı\"))\n    if \"ensemble_logits\" not in ns:\n        return\n    big = [nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)) for _ in range(64)]\n    Xb = torch.randn(256, 16)\n    with torch.no_grad():\n        t_loop = bench(lambda: torch.stack([m(Xb) for m in big]), repeat=3)\n        t_vec = bench(lambda: ns[\"ensemble_logits\"](big, Xb), repeat=3)\n    R.table(\"64 MLP × 256 örnek, ileri geçiş\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for m in models: m(X)\", fmt_time(t_loop), \"1×\"], [\"senin ensemble_logits (vmap)\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]],\n            note=\"Süreye stack_module_state'in parametre kopyası da dahil; gerçek kullanımda bir kez yapılır.\")\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        sizes = [4, 5, 3, 2]\n        Ws = [rng.normal(size=(b, a)) for a, b in zip(sizes[:-1], sizes[1:])]\n        bs = [rng.normal(size=(b, 1)) for b in sizes[1:]]\n        x, y = rng.normal(size=(4, 1)), rng.random((2, 1))\n        sig = lambda z: 1 / (1 + np.exp(-z))\n\n        def C():\n            a = x\n            for W, b in zip(Ws, bs):\n                a = sig(W @ a + b)\n            return 0.5 * np.sum((a - y) ** 2)\n        deltas, dWs, dbs = need(ns, \"backprop\")(Ws, bs, x, y)\n        h = 1e-6\n        c0 = C()\n        for l in range(3):\n            for idx in [(0, 0), (1, 1)]:\n                Ws[l][idx] += h\n                close((C() - c0) / h, dWs[l][idx], atol=1e-5, what=f\"BP4 katman {l}\")\n                Ws[l][idx] -= h\n            bs[l][0, 0] += h\n            close((C() - c0) / h, dbs[l][0, 0], atol=1e-5, what=f\"BP3 katman {l}\")\n            bs[l][0, 0] -= h\n        rows = []\n        for scale in (1.0, 0.1, 0.01):\n            x2 = x * 0 + scale\n            Ws0 = [W.copy() for W in Ws]\n            Ws0[0] = np.zeros_like(Ws0[0])\n            _, dW, db = ns[\"backprop\"](Ws, bs, x2, y)\n            rows.append([scale, f\"{np.abs(dW[0]).mean():.2e}\", f\"{np.abs(db[0]).mean():.2e}\", f\"{np.abs(dW[0]).mean() / np.abs(db[0]).mean():.3f}\"])\n        ensure(float(rows[2][3]) < 0.02, \"küçük girişte ağırlık gradyanı ≪ bias gradyanı\")\n        R_.table(\"İlk katman: giriş aktivasyonları küçüldükçe (BP4)\", [\"giriş değeri\", \"ort. |∂C/∂W|\", \"ort. |∂C/∂b|\", \"oran\"], rows,\n                 note=\"BP4: ∂C/∂w = a_giriş · δ. Girişi ≈ 0 olan bir nöronun (ya da düşük aktivasyonlu bir gizli nöronun) çıkış ağırlıkları çok yavaş öğrenir; bias'ın gradyanı yalnız δ'ya bağlıdır (BP3). Aynı şekilde BP1–BP2'de σ'(z) ≈ 0 (doymuş nöron) δ'yı ve dolayısıyla tüm gelen ağırlıkların öğrenmesini durdurur.\")\n    R_.test(\"BP1–BP4\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"sign_agreement\", \"saturation\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        Z = rng.normal(0, 1.5, (2000, 10))\n        delta = rng.normal(size=(2000, 5))\n        As = 1 / (1 + np.exp(-Z))\n        At = np.tanh(Z)\n        ss, st = need(ns, \"sign_agreement\")(As, delta), ns[\"sign_agreement\"](At, delta)\n        close(ss, 1.0, what=\"sigmoid: girişler hep pozitif → bir nörona giden tüm ağırlık gradyanları aynı işaretli\")\n        ensure(st < 0.01, \"tanh: sıfır merkezli → karışık işaret\")\n        rows = []\n        for s in (1, 2, 4):\n            Zs = rng.normal(0, s, 20000)\n            rows.append([s, f\"{need(ns, 'saturation')(Zs, 'sigmoid'):.3f}\", f\"{ns['saturation'](Zs, 'tanh'):.3f}\"])\n        R_.table(\"Ön-aktivasyon z ~ N(0, s²) iken doymuş nöron oranı (türev < maksimumun %10'u)\", [\"s\", \"sigmoid\", \"tanh\"], rows,\n                 note=f\"Sigmoid'in iki kusuru: (1) çıkışı sıfır merkezli değil (0, 1): sonraki katmanda bir nörona gelen tüm ağırlıkların gradyanı δ'nın işaretini taşır (örneklerin %{ss * 100:.0f}'ında hepsi aynı işaretli) — ağırlıklar ancak zikzak yaparak farklı yönlere gidebilir; (2) türevi en fazla 0.25: her katmanda gradyanı en az 4 kat küçültür, doyma daha erken başlar. tanh sıfır merkezlidir (%{st * 100:.1f}) ve türevi 1'e kadar çıkar, ama o da büyük |z|'de doyar.\")\n    R_.test(\"sigmoid vs tanh\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        res = {}\n        for w0, b0 in ((0.6, 0.9), (2.0, 2.0)):\n            for loss in (\"mse\", \"ce\"):\n                out = need(ns, \"train_neuron\")(w0, b0, 1.0, 0.0, loss, 0.15, 300)\n                res[(w0, loss)] = out\n                rows.append([f\"w = b = {w0}\" if w0 == 2.0 else \"w = 0.6, b = 0.9\", loss.upper(), f\"{out[0]:.3f}\", f\"{out[50]:.3f}\", f\"{out[150]:.3f}\", f\"{out[-1]:.3f}\"])\n        ensure(abs(res[(0.6, \"mse\")][0] - 1 / (1 + np.exp(-1.5))) < 1e-9, \"ilk çıktı\")\n        ensure(res[(2.0, \"mse\")][150] > 0.8 and res[(2.0, \"ce\")][150] < 0.2, \"doymuş başlangıçta MSE takılır, CE hızla öğrenir\")\n        R_.table(\"Nielsen örneği: girdi 1, hedef 0, η = 0.15\", [\"başlangıç\", \"kayıp\", \"adım 0\", \"adım 50\", \"adım 150\", \"adım 300\"], rows,\n                 note=\"Başlangıç çıktısı 0.98 (yanlış ve doymuş) iken MSE'nin δ = (a − y)σ'(z) terimi σ' ≈ 0.02 yüzünden çok küçüktür: insan 'çok yanlışken hızlı öğrenir', bu nöron en yanlışken en yavaş öğrenir. Çapraz entropide σ' sadeleşir: δ = a − y, hata büyükse gradyan büyük. Sigmoid/softmax çıkışlarıyla CE kullanmanın gerekçesi budur.\")\n    R_.test(\"öğrenme yavaşlaması\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m3_2_mlp_backprop as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

İki katmanlı bir MLP (batch hâli, satırlar örnek):

$$
Z_1=XW_1+b_1,\quad H=\text{ReLU}(Z_1),\quad S=HW_2+b_2,\quad L=\text{CE}(S,y).
$$

Geri geçiş her katman için yerel bir kural; M3.1'deki skaler kuralların matris hâli:

| katman | ileri | geri ($dY$ gelen gradyan) |
|---|---|---|
| affine | $Y=XW+b$ | $dX=dY\,W^\top$, $dW=X^\top dY$, $db=\sum_i dY_i$ |
| ReLU | $Y=\max(0,X)$ | $dX=dY\odot[X>0]$ |
| softmax + CE | $L=\frac1n\sum-\log p_{y_i}$ | $dS=\frac1n(P-Y_{\text{onehot}})$ |

CS231n'in tasarımı: her katman `forward(x) → (out, cache)` ve `backward(dout, cache) → gradyanlar`. `cache`, geri geçişte gereken ileri değerleri tutar. Ağın geri geçişi, bu fonksiyonları ters sırada çağırmaktan ibaret.

## 1 · Elle

Önce ileri geçiş: iki gizli nöronun ikisi de pozitif, ReLU ikisini de geçiriyor. Çıktı 3, hedef 1.

Geri geçişte gradyanı çıktıdan geriye doğru taşı. $dh$ iki gizli nörona ulaştığında ReLU maskesi her ikisini de geçiriyor ($z>0$). Bir nöronun $z$'si negatif olsaydı o nörona hiç gradyan akmazdı ("ölü ReLU").

Son olarak $dx$: girdiye göre gradyan, adversarial örneklerin ve saliency haritalarının temeli.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Ağ: $z=W_1x$, $h=\text{ReLU}(z)$, $\hat y=w_2\cdot h$, $L=\tfrac12(\hat y-y)^2$. $x=(1,2)$, $W_1=\begin{bmatrix}1&2\\-1&1\end{bmatrix}$, $w_2=(1,-2)$, $y=1$, bias yok**

- `z1`: $z_1$
- `z2`: $z_2$
- `yhat`: $\hat y$
- `L`: $L$

**Geri geçiş, $\partial L/\partial\hat y=\hat y-y=2$**

- `dw21`: $\partial L/\partial w_{2,1}$
- `dh2`: $\partial L/\partial h_2$
- `dW11`: $\partial L/\partial W_{1,11}$
- `dW22`: $\partial L/\partial W_{1,22}$
- `dx1`: $\partial L/\partial x_1$

In [ ]:
cevaplar = dict(
    z1=None,
    z2=None,
    yhat=None,
    L=None,
    dw21=None,
    dh2=None,
    dW11=None,
    dW22=None,
    dx1=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"z1":[5,0],"z2":[1,0],"yhat":[3,0],"L":[2,0],"dw21":[10,0],"dh2":[-4,0],"dW11":[2,0],"dW22":[-8,0],"dx1":[6,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, **tek örnek**, vektörler liste, matrisler satır listesi ($W$: `out × in`, yani $z=Wx+b$).

- `linear_forward(x, W, b)` → `z`
- `linear_backward(dz, x, W)` → `(dx, dW, db)`
- `relu_forward(z)`, `relu_backward(dh, z)`
- `mlp_loss(x, y, W1, b1, w2, b2)` → `(L, grads)`; $\hat y=w_2\cdot\text{ReLU}(W_1x+b_1)+b_2$, $L=\frac12(\hat y-y)^2$; `grads` sözlüğü: `"W1", "b1", "w2", "b2", "x"`. Önceki fonksiyonları kullan.

In [ ]:
%%writefile stage_b.py
def linear_forward(x, W, b):
    ...


def linear_backward(dz, x, W):
    ...


def relu_forward(z):
    ...


def relu_backward(dh, z):
    ...


def mlp_loss(x, y, W1, b1, w2, b2):
    ...


L, g = mlp_loss([1, 2], 1, [[1, 2], [-1, 1]], [0, 0], [1, -2], 0)
print(L, g)    # 2.0, W1: [[2, 4], [-4, -8]], x: [6, 0]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

CS231n modüler katmanları NumPy'da, **batch**; tüm katman fonksiyonlarında **döngü yok**:

- `affine_forward(x, w, b)` → `(out, cache)`; `affine_backward(dout, cache)` → `(dx, dw, db)`
- `relu_forward(x)`, `relu_backward(dout, cache)`
- `softmax_loss(scores, y)` → `(loss, dscores)`
- `two_layer_loss(X, y, params, reg=0.0)` → `(loss, grads)`; `params`: `W1 (D, H)`, `b1`, `W2 (H, C)`, `b2`; ceza `reg·(‖W1‖² + ‖W2‖²)`.
- `train(X, y, H, C, lr=1.0, steps=5000, reg=1e-4, seed=0)` → `params`; başlatma `0.01 * rng.standard_normal`, tam batch, yalnız adım döngüsü.

Kontrol, CS231n'in spiral verisinde (3 kol) eğitim doğruluğunun ≥ 0.95 olduğunu doğrular. Lineer softmax bu veride ~%50'de kalır.

In [ ]:
%%writefile stage_c.py
import numpy as np


def affine_forward(x, w, b):
    ...


def affine_backward(dout, cache):
    ...


def relu_forward(x):
    ...


def relu_backward(dout, cache):
    ...


def softmax_loss(scores, y):
    ...


def two_layer_loss(X, y, params, reg=0.0):
    ...


def train(X, y, H, C, lr=1.0, steps=5000, reg=1e-4, seed=0):
    ...


def spiral(n=100, K=3, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for k in range(K):
        r = np.linspace(0.0, 1, n)
        t = np.linspace(k * 4, (k + 1) * 4, n) + rng.normal(size=n) * 0.2
        X.append(np.c_[r * np.sin(t), r * np.cos(t)]); y += [k] * n
    return np.concatenate(X), np.array(y)


X, y = spiral()
p = train(X, y, 100, 3)
h = np.maximum(0, X @ p["W1"] + p["b1"])
print("doğruluk:", ((h @ p["W2"] + p["b2"]).argmax(1) == y).mean())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Aynı ağ PyTorch'ta `nn.Sequential(nn.Linear, nn.ReLU, nn.Linear)`. İki görev:

- `two_layer_torch(X, y, W1, b1, W2, b2, reg)` → skaler kayıp tensörü (`F.cross_entropy` serbest). Parametreler `requires_grad=True` geliyor. Kontrol, autograd gradyanlarının c'deki formüllerle aynı olduğunu doğrular.
- `train_mlp(X, y, H, C, epochs=300, lr=0.05)` → `nn.Module`; `Adam`, tam batch.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch import nn


def two_layer_torch(X, y, W1, b1, W2, b2, reg):
    ...


def train_mlp(X, y, H, C, epochs=300, lr=0.05):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

16 küçük ağı (aynı mimari, farklı başlatma) aynı veride çalıştırmak: döngüyle 16 ileri geçiş ya da **tek** ileri geçiş.

```python
from torch.func import stack_module_state, functional_call
params, buffers = stack_module_state(models)        # her parametre (E, …)
base = copy.deepcopy(models[0]).to("meta")          # yalnız yapıyı taşıyan şablon
f = lambda p, b, x: functional_call(base, (p, b), (x,))
logits = torch.vmap(f, in_dims=(0, 0, None))(params, buffers, X)   # (E, N, C)
```

- `ensemble_logits(models, X)` → `(E, N, C)`. **Döngü yok.**
- `ensemble_predict(models, X)` → `(N,)`; olasılıkların ortalamasının argmax'ı.

Kontrol, sonucu modeller üzerinde döngüyle karşılaştırır ve hız tablosu üretir.

In [ ]:
%%writefile stage_e.py
import copy
import torch
from torch import nn
from torch.func import stack_module_state, functional_call


def ensemble_logits(models, X):
    """→ (E, N, C). Döngü yok."""
    ...


def ensemble_predict(models, X):
    """→ (N,): ortalama olasılığın argmax'ı. Döngü yok."""
    ...


torch.manual_seed(0)
models = [nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 3)) for _ in range(16)]
X = torch.randn(100, 2)
print(ensemble_logits(models, X).shape)    # (16, 100, 3)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Dört geri yayılım denklemi (BP1–BP4) ve yavaş öğrenme

Nielsen'in dört geri yayılım denklemi ($z^l=W^la^{l-1}+b^l$, $a^l=\sigma(z^l)$, $\delta^l=\partial C/\partial z^l$):

$$\text{BP1: }\delta^L=\nabla_aC\odot\sigma'(z^L),\quad\text{BP2: }\delta^l=((W^{l+1})^\top\delta^{l+1})\odot\sigma'(z^l),\quad\text{BP3: }\frac{\partial C}{\partial b^l_j}=\delta^l_j,\quad\text{BP4: }\frac{\partial C}{\partial w^l_{jk}}=a^{l-1}_k\delta^l_j.$$

NumPy, tek örnek (sütun vektörleri), sigmoid, $C=\tfrac12\|a^L-y\|^2$.

- `backprop(Ws, bs, x, y)` → `(deltas, dWs, dbs)` — dört denklemi doğrudan kodla (her katman için `dW = δ aᵀ`, `db = δ`).

Ders alıştırması 2, soru 1: bias ve ağırlık güncellemesi arasındaki fark BP3–BP4'te: ağırlık gradyanı girişteki aktivasyonla çarpılır. Girdisi ≈ 0 olan nöronun çıkış ağırlıkları neredeyse hiç öğrenmez ("yavaş öğrenme"); bias'ın böyle bir çarpanı yoktur.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def backprop(Ws, bs, x, y):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Sigmoid vs tanh: sıfır merkezlilik ve doyma (ders alıştırması 2)

Ders alıştırması 2, soru 2 — sigmoid ve tanh'ın türevden bağımsız kusurları.

NumPy.

- `sign_agreement(A, delta)` — bir katmanın ağırlık gradyanı $\partial C/\partial W=\delta a^\top$ (batch toplamı değil, **tek örnek**) için: her örnekte bir çıkış nöronuna giden ağırlıkların gradyanlarının **hepsinin aynı işaretli** olduğu (satır, örnek) çiftlerinin oranı. `A (N, n_in)` girişteki aktivasyonlar, `delta (N, n_out)`. Döngüsüz.
- `saturation(Z, act)` — `act ∈ {"sigmoid", "tanh"}`: türevin en büyük değerinin %10'unun altına düştüğü ön-aktivasyon oranı.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def sign_agreement(A, delta):
    ...


def saturation(Z, act):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · MSE vs çapraz entropi: doymuş çıkışta öğrenme yavaşlaması

Çıkış nöronunda sigmoid + MSE ile sigmoid + çapraz entropi (Nielsen, Bölüm 3): MSE'de $\delta^L=(a-y)\sigma'(z)$ — çıktı yanlış ama doymuşsa ($a\approx1$, $y=0$) gradyan neredeyse 0; CE'de $\delta^L=a-y$.

NumPy.

- `train_neuron(w0, b0, x, y, loss, lr, steps)` — tek nöron, tek örnek; `loss ∈ {"mse", "ce"}`; her adımdaki çıktı listesini döndür.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def train_neuron(w0, b0, x, y, loss, lr, steps):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Rumelhart, Hinton, Williams (1986): geri yayılımın orijinal deneyleri

[Rumelhart, Hinton, Williams — Learning representations by back-propagating errors (Nature 1986)](https://www.nature.com/articles/323533a0)

Makalenin iki deneyini c aşamasındaki NumPy ağınla tekrarla: (1) simetri tespiti — 6 girdili, 2 gizli nöronlu ağın, girdi vektörünün ayna simetrik olup olmadığını öğrenmesi ve öğrenilen ağırlıkların makaledeki Şekil 1'deki gibi "ayna" desenini (eşit büyüklük, zıt işaret, 1:2:4 oranları) alması; (2) 8-3-8 kodlayıcı — gizli katmanın 3 bitlik ikili kod keşfetmesi. Momentum (makale Denk. 9, α = 0.9) ile ve onsuz yakınsama sürelerini karşılaştır.


**Adımlar**

1. Ağırlıkları küçük rastgele değerlerle başlat; simetri kırma gerekliliğini sıfır başlatmayla göster.

**Kabul ölçütleri**

- Simetri ağında gizli ağırlık oranları ≈ 1:2:4 ve aynalı işaretler.
- 8-3-8'de gizli aktivasyonlar 8 farklı köşeye (ikili kod) yakınsar.

## Makale ödevi: Nielsen (2015), Bölüm 2: geri yayılımın matris biçimi ve hız

[Nielsen — Neural Networks and Deep Learning, Ch. 2 'How the backpropagation algorithm works'](http://neuralnetworksanddeeplearning.com/chap2.html)

x1'deki tek örnekli BP1–BP4'ü mini-batch matris biçimine çevir (tüm batch tek matris çarpımlarıyla) ve bölümün sonundaki "neden hızlı" argümanını ölç: parametre sayısı P olan ağda sonlu farklarla gradyan (P + 1 ileri geçiş) ile geri yayılımın (≈ 2 ileri geçiş) süresini P = 10²…10⁵ için karşılaştır.


**Adımlar**

1. Sonlu farkın doğruluğunu h ∈ {10⁻², 10⁻⁴, 10⁻⁶, 10⁻⁸} için çiz: kesme ve yuvarlama hatası dengesi.

**Kabul ölçütleri**

- Matris biçimi, döngülü sürümle aynı gradyanı verir.
- Süre oranı P ile doğrusal büyür.

## Kaynaklar

- [CS231n — Neural Networks Part 1: Setting up the Architecture](https://cs231n.github.io/neural-networks-1/) — MIT
- [CS231n Assignment 2 — Fully-connected nets (affine_forward/backward)](https://cs231n.github.io/assignments2025/assignment2/) — MIT
- [Prince — Understanding Deep Learning, Bölüm 7 (Gradients and initialization)](https://udlbook.github.io/udlbook/) — CC BY-NC-ND
- [PyTorch Tutorials — Model ensembling (torch.func)](https://docs.pytorch.org/tutorials/intermediate/ensembling.html) — BSD-3